# 03 — Class Distribution

## Objective

Characterize annotation counts, images per class, and the number of objects per image.

## Motivation

Class imbalance can bias training and evaluation. This stage measures that imbalance without applying resampling, and excludes the auxiliary `finger` class from the primary coin-class ratios.

## Inputs

- `curation/outputs/02-annotation-audit/annotations_normalized.csv`

## Outputs

- `curation/outputs/03-class-distribution/manifest.yaml`

The aggregated distribution is stored only in the manifest summary.


In [ ]:
# Environment-specific setup
import pathlib
import sys

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../../')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

ROOT = base_folder.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
from curation.common import load_config, prepare_dataset, stage_output_dir

CONFIG = load_config(ROOT)
DATASET = prepare_dataset(ROOT)
IMAGES_DIR = DATASET["images_dir"]
ANNOTATIONS_DIR = DATASET["annotations_dir"]

from curation.common import write_manifest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STAGE = "03-class-distribution"
ANNOTATIONS_PATH = stage_output_dir("02-annotation-audit", ROOT, create=False) / "annotations_normalized.csv"
if not ANNOTATIONS_PATH.is_file():
    raise FileNotFoundError("Run 02-annotation-audit.ipynb first.")

In [ ]:
annotations = pd.read_csv(ANNOTATIONS_PATH, dtype={"label": str})
coin_labels = list(CONFIG["classes"]["coins"])
coins = annotations[annotations["label"].isin(coin_labels)]
instance_counts = coins["label"].value_counts().reindex(coin_labels, fill_value=0)
image_counts = coins.groupby("label")["relative_path"].nunique().reindex(coin_labels, fill_value=0)
objects_per_image = annotations.groupby("relative_path").size()
all_counts = annotations["label"].value_counts().sort_index()

display(pd.DataFrame({"instances": instance_counts, "images": image_counts}))
all_counts.plot.bar(figsize=(9, 4), title="Annotation instances by class")
plt.tight_layout()
plt.show()

In [ ]:
summary = {
    "coin_labels": coin_labels,
    "auxiliary_labels": list(CONFIG["classes"]["auxiliary"]),
    "total_annotations": int(len(annotations)),
    "coin_annotations": int(len(coins)),
    "annotated_images": int(annotations["relative_path"].nunique()),
    "images_with_multiple_annotations": int((objects_per_image > 1).sum()),
    "coin_instance_imbalance_ratio_max_min": float(instance_counts.max() / instance_counts.min()),
    "coin_image_imbalance_ratio_max_min": float(image_counts.max() / image_counts.min()),
    "instance_counts": {str(k): int(v) for k, v in all_counts.items()},
    "coin_image_counts": {str(k): int(v) for k, v in image_counts.items()},
    "objects_per_image": {
        "mean": float(objects_per_image.mean()),
        "median": float(objects_per_image.median()),
        "maximum": int(objects_per_image.max()),
    },
}
write_manifest(
    STAGE,
    "03-class-distribution.ipynb",
    inputs={"annotations_normalized": ANNOTATIONS_PATH},
    parameters={"coin_labels": coin_labels, "exclude_auxiliary_from_imbalance": True},
    artifacts=[],
    summary=summary,
    repo_root=ROOT,
)
summary